# STEP 2 — Baseline Runs (always_slm + always_llm)

**Prerequisite:** Steps 0 and 1 complete. All 3 Kaggle datasets attached.

**What it does:**
- Runs `always_slm_1b`, `always_slm_3b`, `always_llm` on both HotpotQA + MuSiQue
- Collects per-step logit features (for Step 4 learned router training)
- Checkpoints after every (model, policy, dataset) triple — safe to interrupt + resume

**Outputs:**
- `results/baseline_episodes.csv` — episode-level results
- `results/baseline_steps.csv`    — step-level logit features (used to train learned router)

**Estimated run time:** ~5 hours

**IMPORTANT:** Run `session_save` at the end and upload results before closing.

In [ ]:
import sys, gc, json
import torch
import pandas as pd
from pathlib import Path

sys.path.insert(0, '/kaggle/working/slm_agents_paper')
from src.session_init import init
P = init()

from src.models    import load_model, get_model_by_name, SLM_MODELS, LLM_MODEL
from src.datasets  import load_samples, build_bm25_index
from src.routing   import AlwaysSLM, AlwaysLLM
from src.react_env import run_react_episode, episode_to_row, steps_to_rows
from src.metrics   import compute_summary

hotpotqa = load_samples(P['data'] / 'hotpotqa.json')
musique  = load_samples(P['data'] / 'musique.json')
all_datasets = {'hotpotqa': hotpotqa, 'musique': musique}

print(f'HotpotQA: {len(hotpotqa)} | MuSiQue: {len(musique)}')

In [ ]:
# Resume-safe setup
EP_PATH   = P['results'] / 'baseline_episodes.csv'
STEP_PATH = P['results'] / 'baseline_steps.csv'

if EP_PATH.exists():
    ep_df   = pd.read_csv(EP_PATH)
    done    = set(zip(ep_df['policy'], ep_df['slm_name'], ep_df['dataset']))
    # aggregate: one done key covers all questions in that combo
    done_keys = set()
    for (policy, slm, ds), grp in ep_df.groupby(['policy','slm_name','dataset']):
        if len(grp) >= len(all_datasets[ds]):
            done_keys.add((policy, slm, ds))
    print(f'⚡ Resuming — {len(done_keys)} combos fully done')
else:
    ep_df   = pd.DataFrame()
    done_keys = set()

if STEP_PATH.exists():
    step_df = pd.read_csv(STEP_PATH)
else:
    step_df = pd.DataFrame()

all_ep_rows   = ep_df.to_dict('records')   if not ep_df.empty   else []
all_step_rows = step_df.to_dict('records') if not step_df.empty else []

In [ ]:
# Baseline configurations to run
# policy_id, router, slm_config
from src.models import MODEL_REGISTRY

slm_1b = get_model_by_name('Qwen2.5-1.5B')
slm_3b = get_model_by_name('Qwen2.5-3B')
llm    = get_model_by_name('Llama-3.1-8B')

BASELINES = [
    ('always_slm_1b', AlwaysSLM(), slm_1b),
    ('always_slm_3b', AlwaysSLM(), slm_3b),
    ('always_llm',    AlwaysLLM(), llm),
]

for policy_id, router, slm_config in BASELINES:
    for ds_name, samples in all_datasets.items():
        key = (policy_id, slm_config.name, ds_name)
        if key in done_keys:
            print(f'✓ Skip {policy_id}/{slm_config.name}/{ds_name}')
            continue

        print(f'\n── {policy_id} | {slm_config.name} | {ds_name} ({len(samples)} q) ──')

        # For always_llm, the "slm" and "llm" are the same model (LLM)
        if policy_id == 'always_llm':
            slm_model, slm_tok = load_model(llm)
            llm_model, llm_tok = slm_model, slm_tok
        else:
            slm_model, slm_tok = load_model(slm_config)
            llm_model, llm_tok = load_model(llm)

        for i, sample in enumerate(samples):
            ep = run_react_episode(
                sample, slm_model, slm_tok, slm_config,
                llm_model, llm_tok, llm,
                router, policy_id,
            )
            all_ep_rows.extend([episode_to_row(ep)])
            all_step_rows.extend(steps_to_rows(ep))

            if (i + 1) % 10 == 0:
                pd.DataFrame(all_ep_rows).to_csv(EP_PATH, index=False)
                pd.DataFrame(all_step_rows).to_csv(STEP_PATH, index=False)
                sr = sum(r['success'] for r in all_ep_rows if r['policy']==policy_id and r['dataset']==ds_name)
                total = sum(1 for r in all_ep_rows if r['policy']==policy_id and r['dataset']==ds_name)
                print(f'  [{i+1}/{len(samples)}] SR={sr/total:.3f}')

        # Final save for this combo
        pd.DataFrame(all_ep_rows).to_csv(EP_PATH, index=False)
        pd.DataFrame(all_step_rows).to_csv(STEP_PATH, index=False)
        done_keys.add(key)

        # Unload models
        if policy_id != 'always_llm':
            del slm_model, slm_tok
        del llm_model, llm_tok
        gc.collect()
        torch.cuda.empty_cache()
        print(f'  GPU freed')

In [ ]:
# Summary
ep_df = pd.read_csv(EP_PATH)
summary = compute_summary(ep_df)
print('\nBaseline Results:')
print(summary.to_string(index=False))

print('\n✅ STEP 2 COMPLETE')
print('   always_llm SR sets the oracle ceiling')
print('   always_slm SR sets the routing floor')
print('   The gap motivates routing.')

In [ ]:
from src.session_save import save
save(P['results'])
print('⚠️  Upload results_snapshot_*.zip before closing this session')